# Semana 2 — Funciones, clausuras y clases aplicado a Penonomé

Aplica las unidades 17 a 21 del curso (funciones, recursividad, lambda con `filter`, `map` y `reduce`, ámbito y clausuras, clases y objetos) a las afectaciones del sismo que publica SINAPROC, el Censo 2023, el índice de pobreza multidimensional del PNUD y la lista de fuentes del estudio.

[Proyecto](../../README.md) | [Anterior](s01-listas-diccionarios-conjuntos.ipynb) | [Siguiente](s03-pilas-colas-busqueda-hash.ipynb)

In [1]:
def verificar(descripcion, obtenido, esperado):
    """Compara el resultado de un ejercicio con el esperado e informa sin detener el notebook."""
    if obtenido == esperado:
        print(f"Correcto: {descripcion}")
    else:
        print(f"Incorrecto: {descripcion}. Se esperaba {esperado!r} y se obtuvo {obtenido!r}")

Los datos se cargan con los mismos cargadores de la semana 1 (`src/penonome/carga.py`), que devuelven listas de diccionarios.

In [2]:
from penonome.carga import cargar_afectaciones, cargar_ipm, cargar_poblacion, cargar_viviendas

afectaciones = cargar_afectaciones()         # SINAPROC, formato largo: una fila por dato
ipm = cargar_ipm()                           # tabla 03, PNUD
poblacion = cargar_poblacion()               # tabla 01
viviendas = cargar_viviendas("corregimiento")

NIVELES = ("Total", "Moderado", "Parcial", "Leve", "Menor")     # niveles de daño de la EDAN; «Total» es el daño total

def dato_sinaproc(categoria, subcategoria, desagregacion="Total"):
    """Devuelve el valor de una afectación de Coclé, o None si no existe."""
    for fila in afectaciones:
        if (fila["categoria"], fila["subcategoria"], fila["desagregacion"]) == (categoria, subcategoria, desagregacion):
            return fila["valor"]
    return None

print(dato_sinaproc("Cobertura", "Viviendas evaluadas"), dato_sinaproc("Cobertura", "Personas en las viviendas evaluadas"))
verificar("Coclé tenía 2,531 viviendas evaluadas", dato_sinaproc("Cobertura", "Viviendas evaluadas"), 2531)
verificar("un dato que no existe devuelve None", dato_sinaproc("Cobertura", "Puentes"), None)

2531 10199
Correcto: Coclé tenía 2,531 viviendas evaluadas
Correcto: un dato que no existe devuelve None


## 1. Funciones: tasas de afectación

Una función agrupa un cálculo con nombre, parámetros y `return` ([unidad 17](../../../../01-python/semana-02/clase-1-funciones-recursividad-lambda/u17-funciones.ipynb)). Las dos fórmulas básicas de este cuaderno son:

- porcentaje = parte / total x 100 (por ejemplo, viviendas con un nivel de daño entre viviendas evaluadas)
- personas por vivienda = personas en las viviendas evaluadas / viviendas evaluadas

El parámetro `decimales` tiene un valor por defecto (1), y la docstring documenta la función.

**Referencia:** las viviendas y las personas son de la EDAN de SINAPROC al 15-nov-2026 y cubren solo las viviendas que se evaluaron, no todas las de Coclé; por eso no se comparan como si fueran una muestra de la provincia.

In [3]:
def porcentaje(parte, total, decimales=1):
    """Porcentaje que representa una parte del total."""
    return round(parte / total * 100, decimales)


def personas_por_vivienda(personas, viviendas_evaluadas, decimales=1):
    """Promedio de personas por vivienda."""
    return round(personas / viviendas_evaluadas, decimales)


evaluadas = dato_sinaproc("Cobertura", "Viviendas evaluadas")
personas = dato_sinaproc("Cobertura", "Personas en las viviendas evaluadas")
parcial = dato_sinaproc("Viviendas por nivel de daño", "Parcial")

print("Viviendas con daño parcial:", porcentaje(parcial, evaluadas), "%")
print("Con tres decimales (palabra clave):", porcentaje(parcial, evaluadas, decimales=3), "%")
print("Personas por vivienda evaluada:", personas_por_vivienda(personas, evaluadas))
help(porcentaje)

censo_coclé = cargar_viviendas("provincia")[0]
print("Personas por vivienda en el Censo 2023 (Coclé):", personas_por_vivienda(censo_coclé["poblacion"], censo_coclé["viviendas"]))

verificar("el 69.1 % de las viviendas evaluadas tuvo daño parcial", porcentaje(parcial, evaluadas), 69.1)
verificar("las viviendas evaluadas tienen 4.0 personas en promedio", personas_por_vivienda(personas, evaluadas), 4.0)
verificar("los porcentajes de los cinco niveles suman 100",
          round(sum(porcentaje(dato_sinaproc("Viviendas por nivel de daño", n), evaluadas) for n in NIVELES), 1), 100.0)

Viviendas con daño parcial: 69.1 %
Con tres decimales (palabra clave): 69.143 %
Personas por vivienda evaluada: 4.0
Help on function porcentaje in module __main__:

porcentaje(parte, total, decimales=1)
    Porcentaje que representa una parte del total.

Personas por vivienda en el Censo 2023 (Coclé): 3.5
Correcto: el 69.1 % de las viviendas evaluadas tuvo daño parcial
Correcto: las viviendas evaluadas tienen 4.0 personas en promedio
Correcto: los porcentajes de los cinco niveles suman 100


Una función puede devolver varios valores en una tupla, y `*args` recibe una cantidad variable de argumentos. Con eso se comprueban dos sumas de control de los datos de SINAPROC: los cinco niveles de daño suman las viviendas evaluadas, y las personas con discapacidad por nivel suman el total.

**Error común:** dividir entre cero detiene el programa con `ZeroDivisionError`. La función `tasa_segura` lo captura con `try`/`except` (unidad 19) y devuelve `None`.

In [4]:
def resumen_nivel(nivel):
    """Devuelve (% de las viviendas evaluadas, % de las personas vulnerables) de un nivel de daño.

    Personas vulnerables: con discapacidad o mujeres embarazadas, según la EDAN.
    """
    vulnerables = (dato_sinaproc("Atención especial por nivel de daño", nivel, "Personas con discapacidad")
                   + dato_sinaproc("Atención especial por nivel de daño", nivel, "Mujeres embarazadas"))
    total_vulnerables = (dato_sinaproc("Atención especial", "Personas con discapacidad")
                         + dato_sinaproc("Atención especial", "Mujeres embarazadas"))
    return (
        porcentaje(dato_sinaproc("Viviendas por nivel de daño", nivel), evaluadas),
        porcentaje(vulnerables, total_vulnerables),
    )


def sumar(*valores):
    """Suma cualquier cantidad de valores."""
    total = 0
    for valor in valores:
        total += valor
    return total


def tasa_segura(parte, total):
    """Como porcentaje, pero devuelve None si el total es cero."""
    try:
        return round(parte / total * 100, 1)
    except ZeroDivisionError:
        return None


viviendas_total, vulnerables_total = resumen_nivel("Total")
print(f"Daño total: {viviendas_total} % de las viviendas evaluadas y {vulnerables_total} % de las personas vulnerables")
print("Porcentaje con total cero:", tasa_segura(5, 0))

suma_viviendas = sumar(*[dato_sinaproc("Viviendas por nivel de daño", n) for n in NIVELES])
suma_discapacidad = sumar(*[dato_sinaproc("Atención especial por nivel de daño", n, "Personas con discapacidad") for n in NIVELES])
verificar("daño total: 16.4 % de las viviendas y 18.6 % de las personas vulnerables", (viviendas_total, vulnerables_total), (16.4, 18.6))
verificar("los cinco niveles suman las viviendas evaluadas", suma_viviendas, evaluadas)
verificar("la discapacidad por nivel suma el total de Coclé", suma_discapacidad, dato_sinaproc("Atención especial", "Personas con discapacidad"))
verificar("un total de cero devuelve None en lugar de fallar", tasa_segura(1, dato_sinaproc("Otras afectaciones", "Albergues activos")), None)

Daño total: 16.4 % de las viviendas evaluadas y 18.6 % de las personas vulnerables
Porcentaje con total cero: None
Correcto: daño total: 16.4 % de las viviendas y 18.6 % de las personas vulnerables
Correcto: los cinco niveles suman las viviendas evaluadas
Correcto: la discapacidad por nivel suma el total de Coclé
Correcto: un total de cero devuelve None en lugar de fallar


## 2. Recursividad: sumas anidadas y proyección compuesta

Una función recursiva se llama a sí misma y necesita una condición de terminación ([unidad 18](../../../../01-python/semana-02/clase-1-funciones-recursividad-lambda/u18-recursividad.ipynb)). El territorio de Penonomé es una estructura anidada: el distrito contiene corregimientos, y los cinco corregimientos divididos contienen a sus corregimientos nuevos. `suma_recursiva` recorre cualquier profundidad.

In [5]:
por_id = {fila["id_corregimiento"]: fila["pob_2023"] for fila in poblacion}

territorio = {
    "penonome_cabecera": por_id["penonome_cabecera"],
    "canaveral": por_id["canaveral"],
    "cocle": por_id["cocle"],
    "el_coco": por_id["el_coco"],
    "rio_grande": por_id["rio_grande"],
    "chiguiri_arriba": [por_id["chiguiri_arriba"], por_id["candelario_ovalle"]],
    "pajonal": [por_id["pajonal"], por_id["general_victoriano_lorenzo"]],
    "rio_indio": [por_id["rio_indio"], por_id["riecito"]],
    "toabre": [por_id["toabre"], [por_id["boca_de_tucue"], por_id["san_miguel"]]],
    "tulu": [por_id["tulu"], por_id["las_minas"]],
}


def suma_recursiva(estructura):
    """Suma todos los números de una estructura de listas y diccionarios anidados."""
    if isinstance(estructura, (int, float)):        # caso base
        return estructura
    if isinstance(estructura, dict):
        estructura = list(estructura.values())
    if not estructura:                               # lista vacía
        return 0
    return suma_recursiva(estructura[0]) + suma_recursiva(estructura[1:])


print("Toabré con sus dos corregimientos nuevos:", suma_recursiva(territorio["toabre"]))
verificar("la suma recursiva del distrito da 104,326", suma_recursiva(territorio), 104326)

Toabré con sus dos corregimientos nuevos: 11568
Correcto: la suma recursiva del distrito da 104,326


La tasa de crecimiento anual compuesta entre 2010 y 2023 se obtiene de `(P2023 / P2010) ** (1 / 13) - 1`. Proyectar es aplicar esa tasa año tras año: `P(n) = P(n - 1) x (1 + tasa)`, una definición recursiva natural.

**Ampliación (no está en el PDF):** es un ejercicio de cálculo, no una proyección oficial; supone que la tasa de 2010-2023 se mantiene, y el terremoto de octubre de 2026 puede alterarla.

In [6]:
poblacion_2010 = sum(f["pob_2010"] for f in poblacion if f["pob_2010"] is not None)
poblacion_2023 = sum(f["pob_2023"] for f in poblacion)
tasa_anual = (poblacion_2023 / poblacion_2010) ** (1 / 13) - 1


def proyectar(poblacion_inicial, tasa, anios):
    """Población tras `anios` años creciendo a `tasa` por año (recursiva)."""
    if anios == 0:
        return poblacion_inicial
    return proyectar(poblacion_inicial, tasa, anios - 1) * (1 + tasa)


proyeccion_2030 = round(proyectar(poblacion_2023, tasa_anual, 7))
print(f"Tasa anual 2010-2023: {tasa_anual * 100:.2f} %")
print(f"Población de 2030 si se mantiene la tasa: {proyeccion_2030:,}")

verificar("la recursión con 13 años reproduce el censo 2023",
          round(proyectar(poblacion_2010, tasa_anual, 13)), poblacion_2023)
verificar("recursión = fórmula cerrada", proyeccion_2030, round(poblacion_2023 * (1 + tasa_anual) ** 7))

Tasa anual 2010-2023: 1.52 %
Población de 2030 si se mantiene la tasa: 115,953
Correcto: la recursión con 13 años reproduce el censo 2023
Correcto: recursión = fórmula cerrada


## 3. Lambda, `filter`, `map` y `reduce` sobre las afectaciones de SINAPROC

Una expresión `lambda` es una función anónima de una línea; `filter` conserva los elementos que cumplen una condición, `map` transforma cada uno y `reduce` los acumula en un solo valor ([unidad 19](../../../../01-python/semana-02/clase-1-funciones-recursividad-lambda/u19-lambda-filter-map-reduce.ipynb)).

**Referencia:** las filas de la categoría «Edad y sexo» cuentan personas en las viviendas evaluadas (corte del 15-nov-2026) y deben sumar el total de personas evaluadas. Las de «Otras afectaciones» son del corte del 25-nov-2026 y no se mezclan con las anteriores.

In [7]:
from functools import reduce

edad_sexo = list(filter(lambda f: f["categoria"] == "Edad y sexo", afectaciones))
hombres = list(filter(lambda f: f["desagregacion"] == "Hombres", edad_sexo))
mujeres = list(filter(lambda f: f["desagregacion"] == "Mujeres", edad_sexo))

grupos = list(map(lambda f: f["subcategoria"], hombres))
pct_mujeres = list(map(lambda par: round(par[1]["valor"] / (par[0]["valor"] + par[1]["valor"]) * 100, 1), zip(hombres, mujeres)))
for grupo, pct in zip(grupos, pct_mujeres):
    print(f"{grupo:<16}{pct:>6.1f} % mujeres")

con_mas_mujeres = [grupo for grupo, pct in filter(lambda par: par[1] > 50, zip(grupos, pct_mujeres))]
total_personas = reduce(lambda acumulado, f: acumulado + f["valor"], edad_sexo, 0)
total_mujeres = reduce(lambda acumulado, f: acumulado + f["valor"], mujeres, 0)

print("Grupos con más mujeres que hombres:", con_mas_mujeres)
print(f"Personas por edad y sexo: {total_personas:,} | mujeres: {total_mujeres:,} ({total_mujeres / total_personas * 100:.1f} %)")

verificar("hombres y mujeres tienen los mismos grupos de edad", [f["subcategoria"] for f in mujeres], grupos)
verificar("la suma por edad y sexo da las personas evaluadas", total_personas, dato_sinaproc("Cobertura", "Personas en las viviendas evaluadas"))
verificar("hay más mujeres que hombres solo en adolescencia y adultos", con_mas_mujeres, ["Adolescencia", "Adultos"])
verificar("5,088 de las personas evaluadas son mujeres", total_mujeres, 5088)

Recién nacidos    46.7 % mujeres
Niñez             45.1 % mujeres
Adolescencia      50.2 % mujeres
Adultos           51.9 % mujeres
Adultos mayores   49.7 % mujeres
Grupos con más mujeres que hombres: ['Adolescencia', 'Adultos']
Personas por edad y sexo: 10,199 | mujeres: 5,088 (49.9 %)
Correcto: hombres y mujeres tienen los mismos grupos de edad
Correcto: la suma por edad y sexo da las personas evaluadas
Correcto: hay más mujeres que hombres solo en adolescencia y adultos
Correcto: 5,088 de las personas evaluadas son mujeres


Las mismas funciones trabajan sobre otras filas. `sorted` con `key=lambda` ordena los grupos de edad y sexo por número de personas, y con las filas del Censo, `reduce` suma la población de los corregimientos con más de la mitad de las viviendas sin internet.

In [8]:
mayores = sorted(edad_sexo, key=lambda f: f["valor"], reverse=True)[:3]
print("Grupos con más personas evaluadas:", [(f["subcategoria"], f["desagregacion"], f["valor"]) for f in mayores])

desconectados = list(filter(lambda f: f["sin_internet"] / f["viviendas"] > 0.5, viviendas))
nombres = list(map(lambda f: f["area"], desconectados))
habitantes = reduce(lambda total, f: total + f["poblacion"], desconectados, 0)

print(f"{len(nombres)} corregimientos con más de la mitad sin internet: {nombres}")
print(f"Viven allí {habitantes:,} personas ({habitantes / 104326 * 100:.1f} % del distrito)")

verificar("el grupo más numeroso es el de mujeres adultas", (mayores[0]["subcategoria"], mayores[0]["desagregacion"]), ("Adultos", "Mujeres"))
verificar("diez corregimientos tienen más de la mitad de viviendas sin internet", len(nombres), 10)

Grupos con más personas evaluadas: [('Adultos', 'Mujeres', 2720), ('Adultos', 'Hombres', 2521), ('Niñez', 'Hombres', 1090)]
10 corregimientos con más de la mitad sin internet: ['Chiguirí Arriba', 'Pajonal', 'Río Indio', 'Toabré', 'Tulú', 'Boca de Tucué', 'Candelario Ovalle', 'Las Minas', 'Riecito', 'San Miguel']
Viven allí 40,290 personas (38.6 % del distrito)
Correcto: el grupo más numeroso es el de mujeres adultas
Correcto: diez corregimientos tienen más de la mitad de viviendas sin internet


## 4. Ámbito y clausuras

Python busca un nombre en el orden **LEGB**: local, función envolvente, global y *built-in* ([unidad 20](../../../../01-python/semana-02/clase-2-ambito-clausuras-y-clases/u20-ambito-y-clausuras.ipynb)). Para reasignar una variable global dentro de una función hace falta `global`; para reasignar una de la función envolvente, `nonlocal`.

In [9]:
DISTRITO = "Penonomé"          # constante global: se lee, no se reasigna
consultas = 0                  # variable global que sí se modifica


def describir(corregimiento):
    """Lee la constante global y modifica el contador global con `global`."""
    global consultas
    consultas += 1
    mensaje = f"{corregimiento} pertenece al distrito de {DISTRITO}"   # variable local
    return mensaje


for nombre in ("El Coco", "Riecito", "Tulú"):
    print(describir(nombre))

print("Consultas realizadas:", consultas)
print("¿'mensaje' existe fuera de la función?", "mensaje" in globals())
verificar("el contador global registró tres consultas", consultas, 3)

El Coco pertenece al distrito de Penonomé
Riecito pertenece al distrito de Penonomé
Tulú pertenece al distrito de Penonomé
Consultas realizadas: 3
¿'mensaje' existe fuera de la función? False
Correcto: el contador global registró tres consultas


Una **clausura** es una función interna que recuerda las variables de la función que la creó, aunque esta ya haya terminado. `crear_tasa(base)` es una fábrica: devuelve una función que expresa una cantidad "por cada `base`". Con `base=100` da porcentajes; con `base=1000`, tasas por mil habitantes, como las que usa la demografía.

In [10]:
def crear_tasa(base):
    """Fábrica de funciones: devuelve una tasa por cada `base` unidades."""
    def tasa(parte, total):
        return round(parte / total * base, 1)
    return tasa


porcentaje = crear_tasa(100)
por_mil = crear_tasa(1000)

tulu = next(f for f in viviendas if f["id_corregimiento"] == "tulu")
print("Tulú, viviendas sin internet:", porcentaje(tulu["sin_internet"], tulu["viviendas"]), "%")
print("Tulú, analfabetas por cada mil habitantes:", por_mil(tulu["analfabeta"], tulu["poblacion"]))
print("Valor guardado en la clausura:", por_mil.__closure__[0].cell_contents)

distrito_por_mil = por_mil(sum(f["analfabeta"] for f in viviendas), sum(f["poblacion"] for f in viviendas))
print("Distrito, analfabetas por mil:", distrito_por_mil)

verificar("Tulú tiene 44.6 analfabetas por cada mil habitantes", por_mil(tulu["analfabeta"], tulu["poblacion"]), 44.6)
verificar("la clausura recuerda base=1000", por_mil.__closure__[0].cell_contents, 1000)

Tulú, viviendas sin internet: 99.4 %
Tulú, analfabetas por cada mil habitantes: 44.6
Valor guardado en la clausura: 1000
Distrito, analfabetas por mil: 15.6
Correcto: Tulú tiene 44.6 analfabetas por cada mil habitantes
Correcto: la clausura recuerda base=1000


Con `nonlocal` la clausura puede **modificar** su estado: `crear_acumulador` devuelve una función que suma cada valor nuevo a un total que solo ella ve. Sirve para un total acumulado sin variables globales. La versión con `lambda` muestra que una clausura también puede ser una expresión de una línea.

**Error común:** sin `nonlocal`, `total += valor` produce `UnboundLocalError`, porque Python trata `total` como local de la función interna.

In [11]:
def crear_acumulador():
    """Devuelve una función que acumula los valores que recibe."""
    total = 0
    def agregar(valor):
        nonlocal total
        total += valor
        return total
    return agregar


acumular = crear_acumulador()
ordenados = sorted(poblacion, key=lambda f: f["pob_2023"], reverse=True)
for fila in ordenados[:5]:
    print(f"{fila['corregimiento']:<28}{acumular(fila['pob_2023']):>8,}")
for fila in ordenados[5:]:
    acumulado = acumular(fila["pob_2023"])

crear_densidad = lambda superficie: (lambda habitantes: round(habitantes / superficie, 1))
densidad_el_coco = crear_densidad(148.3)

verificar("el acumulador termina en 104,326", acumulado, 104326)
verificar("densidad de El Coco con la clausura lambda", densidad_el_coco(11064), 74.6)

Penonomé (cabecera)           25,836
El Coco                       36,900
Cañaveral                     46,224
General Victoriano Lorenzo    54,720
Pajonal                       62,398
Correcto: el acumulador termina en 104,326
Correcto: densidad de El Coco con la clausura lambda


## 5. Clases y objetos: `Corregimiento` y `Fuente`

Una clase agrupa datos (atributos) y comportamiento (métodos); `__init__` construye cada instancia y `self` es la instancia misma ([unidad 21](../../../../01-python/semana-02/clase-2-ambito-clausuras-y-clases/u21-clases-y-objetos.ipynb)). Las clases del proyecto están en `src/penonome/modelos.py`. Este es el núcleo de `Corregimiento`:

In [12]:
import inspect

from penonome.modelos import Corregimiento, CorregimientoNuevo, Fuente, crear_corregimientos, crear_fuentes

codigo = inspect.getsource(Corregimiento)
print(codigo[: codigo.index("    def porcentaje_viviendas")])

class Corregimiento:
    """Un corregimiento del distrito de Penonomé con sus datos censales."""

    distrito = "Penonomé"  # variable de clase: la comparten todas las instancias
    total_creados = 0

    def __init__(self, id_corregimiento, nombre, superficie_km2, poblacion, viviendas=None, ipm=None):
        self.id_corregimiento = id_corregimiento
        self.nombre = nombre
        self.superficie_km2 = superficie_km2
        self.poblacion = dict(poblacion)  # {anio: personas}, solo los censos con dato
        self.viviendas = dict(viviendas or {})  # columnas de la tabla 02
        self.ipm = dict(ipm or {})  # columnas de la tabla 03 (pobreza multidimensional, PNUD)
        Corregimiento.total_creados += 1

    def poblacion_en(self, anio=2023):
        """Población del censo `anio`, o None si no hay dato."""
        return self.poblacion.get(anio)

    def densidad(self, anio=2023):
        """Habitantes por km² en el censo `anio`, redondeado a un decimal."""
        habitan

`crear_corregimientos()` une las tablas 01, 02 y 03 (Censo e IPM) por `id_corregimiento` y crea un objeto por fila. `__str__` define lo que muestra `print`; `__repr__`, lo que se ve en una lista; `__lt__` permite ordenar con `sorted`. `distrito` y `total_creados` son **variables de clase**: las comparten todas las instancias.

In [13]:
corregimientos = crear_corregimientos()
el_coco = next(c for c in corregimientos if c.id_corregimiento == "el_coco")

print(el_coco)                                  # __str__
print(corregimientos[:3])                       # __repr__
print("Más poblado:", max(corregimientos))      # usa __lt__
print("Distrito (variable de clase):", el_coco.distrito, "| Objetos creados:", Corregimiento.total_creados)
print("Atributos de la instancia:", list(el_coco.__dict__))
print(f"Densidad {el_coco.densidad()} hab/km², crecimiento {el_coco.crecimiento()} %, "
      f"sin internet {el_coco.porcentaje_viviendas('sin_internet')} %")
print(f"Analfabetismo {el_coco.tasa_analfabetismo()} %, pobreza multidimensional {el_coco.incidencia_pobreza()} %")

verificar("se crearon 16 objetos", Corregimiento.total_creados, 16)
verificar("El Coco creció 97.4 % entre 2010 y 2023", el_coco.crecimiento(), 97.4)
verificar("El Coco tiene 12 % de pobreza multidimensional", el_coco.incidencia_pobreza(), 12)
verificar("el más poblado es la cabecera", max(corregimientos).nombre, "Penonomé (cabecera)")

El Coco: 11,064 habitantes en 2023
[Corregimiento('penonome_cabecera'), Corregimiento('canaveral'), Corregimiento('cocle')]
Más poblado: Penonomé (cabecera): 25,836 habitantes en 2023
Distrito (variable de clase): Penonomé | Objetos creados: 16
Atributos de la instancia: ['id_corregimiento', 'nombre', 'superficie_km2', 'poblacion', 'viviendas', 'ipm']
Densidad 74.6 hab/km², crecimiento 97.4 %, sin internet 19.4 %
Analfabetismo 1.0 %, pobreza multidimensional 12 %
Correcto: se crearon 16 objetos
Correcto: El Coco creció 97.4 % entre 2010 y 2023
Correcto: El Coco tiene 12 % de pobreza multidimensional
Correcto: el más poblado es la cabecera


**Error común:** la densidad de la cabecera calculada con la superficie redondeada (25,836 / 53.4 = 483.8) no coincide con la publicada (484.2). El INEC divide entre la superficie sin redondear; por eso se usa el dato publicado y no se recalcula.

In [14]:
cabecera = corregimientos[0]
densidad_publicada = next(f["densidad_2023"] for f in poblacion if f["id_corregimiento"] == "penonome_cabecera")
print("Calculada:", cabecera.densidad(), "| Publicada:", densidad_publicada)
verificar("la diferencia es de 0.4 hab/km²", round(densidad_publicada - cabecera.densidad(), 1), 0.4)

Calculada: 483.8 | Publicada: 484.2
Correcto: la diferencia es de 0.4 hab/km²


La herencia permite reutilizar una clase: `CorregimientoNuevo` hereda de `Corregimiento` todo lo anterior y añade el atributo `padre`. Además **sobrescribe** `crecimiento`, porque un corregimiento que no existía en 2010 no tiene un crecimiento propio. Para medirlo hay que sumarlo con su corregimiento original.

In [15]:
nuevos = [c for c in corregimientos if isinstance(c, CorregimientoNuevo)]
print(nuevos)

gvl = next(c for c in nuevos if c.id_corregimiento == "general_victoriano_lorenzo")
pajonal = next(c for c in corregimientos if c.id_corregimiento == gvl.padre)
bloque = round(((pajonal.poblacion_en(2023) + gvl.poblacion_en(2023)) / pajonal.poblacion_en(2010) - 1) * 100, 1)

print("Crecimiento de GVL por sí solo:", gvl.crecimiento())
print("Pajonal solo:", pajonal.crecimiento(), "% | Pajonal + GVL:", bloque, "%")
print("¿GVL también es un Corregimiento?", isinstance(gvl, Corregimiento))
print("Pobreza multidimensional, Pajonal y GVL:", pajonal.incidencia_pobreza(), gvl.incidencia_pobreza())

verificar("hay seis corregimientos nuevos", len(nuevos), 6)
verificar("Pajonal y GVL juntos crecieron 19.2 %", bloque, 19.2)
verificar("GVL hereda los métodos de Corregimiento", (pajonal.incidencia_pobreza(), gvl.incidencia_pobreza()), (45, 28))

[CorregimientoNuevo('boca_de_tucue', padre='toabre'), CorregimientoNuevo('candelario_ovalle', padre='chiguiri_arriba'), CorregimientoNuevo('general_victoriano_lorenzo', padre='pajonal'), CorregimientoNuevo('las_minas', padre='tulu'), CorregimientoNuevo('riecito', padre='rio_indio'), CorregimientoNuevo('san_miguel', padre='toabre')]
Crecimiento de GVL por sí solo: None
Pajonal solo: -43.4 % | Pajonal + GVL: 19.2 %
¿GVL también es un Corregimiento? True
Pobreza multidimensional, Pajonal y GVL: 45 28
Correcto: hay seis corregimientos nuevos
Correcto: Pajonal y GVL juntos crecieron 19.2 %
Correcto: GVL hereda los métodos de Corregimiento


La clase `Fuente` modela cada fila de `fuentes.csv`, la lista de fuentes del estudio. Su variable de clase `TIPOS_OFICIALES` define qué cuenta como fuente oficial, y sus métodos responden preguntas sobre la calidad de la evidencia.

In [16]:
fuentes = crear_fuentes()
oficiales = [f for f in fuentes if f.es_oficial()]
verificadas = [f for f in oficiales if f.esta_verificada()]
dominios_inec = [f for f in oficiales if f.dominio() == "www.inec.gob.pa"]

print("Fuentes:", len(fuentes), "| oficiales u organismos:", len(oficiales), "| oficiales verificadas:", len(verificadas))
print("Ejemplo:", fuentes[0])
print(repr(fuentes[0]))
print("Fuentes del INEC:", len(dominios_inec))

import csv
from penonome import FUENTES
with open(FUENTES, encoding="utf-8-sig", newline="") as archivo:
    filas_csv = sum(1 for _ in csv.DictReader(archivo))
verificar("se crea un objeto Fuente por cada fila de fuentes.csv", len(fuentes), filas_csv)
verificar("las oficiales son un subconjunto de todas las fuentes", all(f in fuentes and f.es_oficial() for f in oficiales), True)

Fuentes: 113 | oficiales u organismos: 29 | oficiales verificadas: 17
Ejemplo: [F001] INEC, Censo 2023: P0705547520231109105354CUADRO 10
Fuente('F001', tipo='oficial')
Fuentes del INEC: 6
Correcto: se crea un objeto Fuente por cada fila de fuentes.csv
Correcto: las oficiales son un subconjunto de todas las fuentes


## Conclusiones

- Las funciones `porcentaje` y `personas_por_vivienda` resumen la EDAN de SINAPROC: **2,531 viviendas** evaluadas en Coclé con **10,199 personas** (4.0 por vivienda), de ellas **1,750 (69.1 %)** con daño parcial y 414 (16.4 %) con daño total. Los cinco niveles suman las viviendas evaluadas y sus porcentajes suman 100.
- Entre las personas con discapacidad o embarazadas, el **18.6 %** vive en viviendas con daño total, frente al 16.4 % de las viviendas. Las dos cifras no se comparan de forma directa porque cuentan unidades distintas (personas y viviendas).
- La suma recursiva del territorio confirma los **104,326** habitantes del distrito. Si se mantuviera la tasa anual de 2010-2023 (**1.52 %**), Penonomé rondaría los 116,000 habitantes en 2030; es un ejercicio, no una proyección oficial.
- Con `filter`, `map` y `reduce`, las filas de edad y sexo suman las 10,199 personas evaluadas; las mujeres son el **49.9 %** y solo superan a los hombres en adolescencia y en adultos. El grupo más numeroso es el de mujeres adultas (2,720).
- Diez de los dieciséis corregimientos tienen más de la mitad de sus viviendas sin internet, y allí viven **40,290** personas (38.6 % del distrito). La clausura `crear_tasa(1000)` muestra la misma brecha en educación: Tulú tiene 44.6 analfabetas por cada mil habitantes.
- Con clases, la regla de la Ley 296 queda dentro del modelo: un `CorregimientoNuevo` no informa un crecimiento engañoso, y Pajonal junto con General Victoriano Lorenzo creció **19.2 %**, en lugar de la caída aparente de 43.4 %. Cada objeto también guarda su incidencia de pobreza multidimensional: 45 % en Pajonal y 28 % en General Victoriano Lorenzo.

[Proyecto](../../README.md) | [Anterior](s01-listas-diccionarios-conjuntos.ipynb) | [Siguiente](s03-pilas-colas-busqueda-hash.ipynb)